In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/home-data-for-ml-course/sample_submission.csv
/kaggle/input/competitions/home-data-for-ml-course/sample_submission.csv.gz
/kaggle/input/competitions/home-data-for-ml-course/train.csv.gz
/kaggle/input/competitions/home-data-for-ml-course/data_description.txt
/kaggle/input/competitions/home-data-for-ml-course/test.csv.gz
/kaggle/input/competitions/home-data-for-ml-course/train.csv
/kaggle/input/competitions/home-data-for-ml-course/test.csv


In [2]:
import pandas as pd
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

train = pd.read_csv("/kaggle/input/competitions/home-data-for-ml-course/train.csv")
train.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [3]:
test = pd.read_csv("/kaggle/input/competitions/home-data-for-ml-course/test.csv")
test.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,1461,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,AllPub,...,120,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal
1,1462,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal
2,1463,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal
3,1464,60,RL,78.0,9978,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,6,2010,WD,Normal
4,1465,120,RL,43.0,5005,Pave,NaN,IR1,HLS,AllPub,...,144,0,NaN,NaN,NaN,0,1,2010,WD,Normal


In [4]:
# apply log transformation bcoz its a case of right skew where we have many moderately priced house and few extremely priced

train['SalePrice'] = np.log1p(train['SalePrice'])


#For columns where null means absence:Fill with "None"

X = train.drop(["SalePrice", "Id"], axis=1)
y = train["SalePrice"]

X_test_final = test.drop(["Id"], axis=1)

# handle missing categorical columns 
cat_cols = X.select_dtypes(include="object").columns

for col in cat_cols:
    X[col] = X[col].fillna("None")
    X_test_final[col] = X_test_final[col].fillna("None")


# handle missing numerical columns
num_cols = X.select_dtypes(exclude="object").columns

for col in num_cols:
    median_val = X[col].median()

    X[col] = X[col].fillna(median_val)
    X_test_final[col] = X_test_final[col].fillna(median_val)

In [5]:
# apply one hot encoding 

X = pd.get_dummies(X)
X_test_final = pd.get_dummies(X_test_final)

In [6]:
# to avoid train and test having different columns
X, X_test_final = X.align(
    X_test_final,
    join="left",
    axis=1,
    fill_value=0
)

In [7]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [8]:
# from sklearn.linear_model import LinearRegression
# from sklearn.metrics import mean_squared_error

# lr_model = LinearRegression()

# lr_model.fit(X_train, y_train)

# val_preds = lr_model.predict(X_val)

In [9]:
# rmse = np.sqrt(mean_squared_error(y_val, val_preds))

# print("Linear Regression RMSE:", rmse)

In [10]:
# NOW LETS TRY RANDOM FOREST

In [11]:
# from sklearn.ensemble import RandomForestRegressor
# from sklearn.metrics import mean_squared_error
# import numpy as np

# rf_model = RandomForestRegressor(
#     n_estimators=200,
#     random_state=42
# )

# rf_model.fit(X_train, y_train)
# rf_preds = rf_model.predict(X_val)
# rf_rmse = np.sqrt(mean_squared_error(y_val, rf_preds))
# print(rf_rmse)

In [12]:
# NOW LETS TRY XGBOOST

In [13]:
# from xgboost import XGBRegressor
# from sklearn.metrics import mean_squared_error
# import numpy as np

# xgb_model = XGBRegressor(
#     n_estimators=500,
#     learning_rate=0.05,
#     max_depth=3,
#     subsample=0.8,
#     colsample_bytree=0.8,
#     random_state=42
# )

# xgb_model.fit(X_train, y_train)
# xgb_preds = xgb_model.predict(X_val)
# xgb_rmse = np.sqrt(mean_squared_error(y_val, xgb_preds))
# print(xgb_rmse)

In [14]:
# XGBoost with gridSearch

In [15]:
xgb = XGBRegressor(
    random_state=42
)


param_grid = {
    "n_estimators": [200, 500],
    "max_depth": [3, 5],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}

grid_search = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    scoring="neg_root_mean_squared_error",
    cv=5,
    verbose=1,
    n_jobs=-1
)

grid_search.fit(X_train, y_train)
print("Best Parameters:")
print(grid_search.best_params_)

print("Best CV Score:")
print(grid_search.best_score_)


best_xgb = grid_search.best_estimator_

xgb_preds = best_xgb.predict(X_val)

rmse = np.sqrt(mean_squared_error(y_val, xgb_preds))

print("Validation RMSE:", rmse)

best_xgb.fit(X, y)
test_preds = best_xgb.predict(X_test_final)

final_preds = np.expm1(test_preds)
output = pd.DataFrame({
    "Id": test["Id"],
    "SalePrice": final_preds
})

output.to_csv("submission.csv", index=False)

print("Submission file created successfully!")



Fitting 5 folds for each of 48 candidates, totalling 240 fits
Best Parameters:
{'colsample_bytree': 1.0, 'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 500, 'subsample': 0.8}
Best CV Score:
-0.12525994765485932
Validation RMSE: 0.12813581777234914
Submission file created successfully!
